In [3]:
from typing import TypedDict,Annotated,List,Literal,Dict,Any
from langchain_core.messages import BaseMessage,AIMessage,HumanMessage,SystemMessage
from langgraph.graph import START,StateGraph,END,MessagesState
from langgraph.checkpoint.memory import MemorySaver
import random
from datetime import datetime



BaseMessage ek common contract/wrapper hai jo LLM chat history ko clean,
 structured aur flexible banata hai taaki LangChain/LangGraph smoothly handle kar sake.

Flow Summary (Yeh sab milkar kaise kaam karte hain):

User request aayi ➔ workflow_stage = 'planning'.

Supervisor ne plan banaya ➔ task_assignment update hua.\

Supervisor ne choose kiya current_agent = 'researcher'.

Researcher ne kaam kiya ➔ uska result agent_outputs['researcher'] mein save hua.
iteration_count badh gaya (+1).

Loop chala jab tak kaam complete na ho ya max_iteration hit na ho.

End mein supervisor ne summary banakar final_output mein daal di aur user ko bhej di.

In [ ]:
class superviserState(MessagesState):
    """ State for supervisor multi agent system"""
    current_agent:str =''
    task_assignment:Dict[str,List[str]]={} # for tracking what each agent should do
    agent_outputs: Dict[str,Any]={} # store output from each agent
 
    workflow_stage:str='initial' # Track workflow progress

    iteration_count:int =0

    max_iteration:int=10
    final_output:str=''


In [8]:
class superviserState(MessagesState):
    """ State  multi agent system"""
    next_agent:str =''
    research_data:str=""
    analysis:str=""
    final_report:str=''
    task_complete:bool=False
    current_task:str=""

In [11]:
import os
from dotenv import load_dotenv

# Load environment variables from .env
load_dotenv()

print("Tracing Enabled:", os.getenv("LANGCHAIN_TRACING_V2"))
print("LangSmith Project:", os.getenv("LANGCHAIN_PROJECT"))
print("Groq API Key Configured:", bool(os.getenv("GROQ_API_KEY")))

Tracing Enabled: true
LangSmith Project: GenAIAPPW/TOPENAI
Groq API Key Configured: True


In [12]:
from langchain.chat_models import init_chat_model

# Initialize Chat Model dynamically with Groq backend
llm = init_chat_model(model='openai/gpt-oss-20b', model_provider='groq')
print("Initialized model:", llm)

Initialized model: metadata={'lc_versions': {'langchain-core': '1.6.4', 'langchain': '1.4.2'}} profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True} client=<groq.resources.chat.completions.Completions object at 0x000001D1418A7620> async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001D141A481A0> model_name='openai/gpt-oss-20b' model_kwargs={} groq_api_key=SecretStr('**********')


In [13]:
from langchain_core.prompts import ChatPromptTemplate


def create_supervisor_chain():
    """
    Creates the supervisor decision chain.
    """

    supervisor_prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                """
                You are a supervisor managing a team of agents:

                1. Researcher - Gathers information and data
                2. Analyst - Analyzes data and provides insights
                3. Writer - Creates reports and summaries

                Based on the current state and conversation,
                decide which agent should work next.

                If the task is complete, respond with 'DONE'.

                Current state:
                - Has research data: {has_research}
                - Has analysis: {has_analysis}
                - Has report: {has_report}

                Respond with ONLY the agent name
                (researcher/analyst/writer) or 'DONE'.
                """,
            ),
            ("human", "{task}"),
        ]
    )

    return supervisor_prompt | llm